<a href="https://colab.research.google.com/github/pranavlandge36/ml-learning-notebooks/blob/main/Pytorch-Learning/QA_System_Pytorch_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df=pd.read_csv('100_Unique_QA_Dataset.csv')

In [2]:
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [4]:
df.shape

(90, 2)

In [5]:
## TOKENIZATION
def tokenize(text):
  text=text.lower()
  text= text.replace('?','')
  text= text.replace("'",'')
  return text.split()



In [7]:
tokenize(df['question'][0])

['what', 'is', 'the', 'capital', 'of', 'france']

In [15]:
# VOCABULARY BUILDING
vocab={'<UNK>':0}

def build_vocab(row):
  tok_que = tokenize(row['question'])
  tok_ans = tokenize(row['answer'])

  merged_token= tok_que+tok_ans
  for tokens in merged_token:
    if tokens not in vocab:
      vocab[tokens]=len(vocab)


In [16]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [18]:
len(vocab)

324

In [19]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [25]:
# TEXT TO NUMERICAL INTEGERS/INDICES

def text_to_num(text,vocab):
  indexed_text=[]
  for tokens in tokenize(text):
    if tokens in vocab:
      indexed_text.append(vocab[tokens])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [27]:
text_to_num('What is your name ?',vocab)

[1, 2, 0, 0]

In [28]:
import torch
from torch.utils.data import Dataset, DataLoader

In [29]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return df.shape[0]
  def __getitem__(self, index):
    numerical_question= text_to_num(self.df.iloc[index]['question'],self.vocab)
    numerical_answer= text_to_num(self.df.iloc[index]['answer'],self.vocab)
    return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [30]:
dataset=QADataset(df,vocab)

In [32]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [33]:
dataloader= DataLoader(dataset,batch_size=1,shuffle=True)

## SKIPPED PADDING AS BATCH SIZE=1

In [34]:
import torch.nn as nn

In [46]:
class simpleRNN(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding= nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64,batch_first=True)
    self.fc= nn.Linear(64,vocab_size)

  def forward(self,question):
      embedded_question=self.embedding(question)
      hidden, final=self.rnn(embedded_question)
      final_output=final[-1]
      output=self.fc(final_output)
      return output

In [47]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [48]:
learning_rate = 0.001
epochs = 50
model = simpleRNN(len(vocab))
optimizer= torch.optim.Adam(model.parameters(),lr=learning_rate)
criterion= nn.CrossEntropyLoss()

In [49]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 521.215254
Epoch: 2, Loss: 449.399967
Epoch: 3, Loss: 368.375893
Epoch: 4, Loss: 308.977504
Epoch: 5, Loss: 257.741335
Epoch: 6, Loss: 211.096303
Epoch: 7, Loss: 169.310707
Epoch: 8, Loss: 132.152633
Epoch: 9, Loss: 102.721293
Epoch: 10, Loss: 78.538781
Epoch: 11, Loss: 61.231781
Epoch: 12, Loss: 48.213804
Epoch: 13, Loss: 38.660499
Epoch: 14, Loss: 31.103487
Epoch: 15, Loss: 25.644355
Epoch: 16, Loss: 21.329814
Epoch: 17, Loss: 18.300304
Epoch: 18, Loss: 15.952291
Epoch: 19, Loss: 13.939460
Epoch: 20, Loss: 12.281402
Epoch: 21, Loss: 10.919035
Epoch: 22, Loss: 9.773571
Epoch: 23, Loss: 8.696731
Epoch: 24, Loss: 7.924635
Epoch: 25, Loss: 7.078369
Epoch: 26, Loss: 6.457947
Epoch: 27, Loss: 5.615977
Epoch: 28, Loss: 5.049846
Epoch: 29, Loss: 4.589984
Epoch: 30, Loss: 4.227256
Epoch: 31, Loss: 3.870265
Epoch: 32, Loss: 3.510651
Epoch: 33, Loss: 3.219316
Epoch: 34, Loss: 2.978319
Epoch: 35, Loss: 2.753043
Epoch: 36, Loss: 2.547131
Epoch: 37, Loss: 2.368730
Epoch: 38, Loss: 

In [54]:
def predict(model,question,threshold=0.5):

  numerial_question= text_to_num(question,vocab)

  question_tensor= torch.tensor(numerial_question).unsqueeze(0)

  output= model(question_tensor)
  output_prob= torch.softmax(output,dim=1)
  value, index = torch.max(output_prob, dim=1)
  if value > threshold:
    return print(list(vocab.keys())[index])
  else:
    return ('i dont know')


In [55]:
predict(model,'what is capital of france ?')

paris


In [56]:
predict(model,'ho painted the Mona Lisa?')

leonardo-da-vinci
